# Built-in Data Structures from Scratch

## Lists, Arrays & Slicing Internals

### Core Mechanics & Theory

To write high-performance Python and prepare for low-level tensor/C-array manipulation, you need to understand how Python lists work in memory.

### 1. Lists are Dynamic Arrays of Pointers

A Python list is not a linked list and not a flat array of raw data. It is a C array of pointers to PyObjects:

```
PyObject** ob_item
list variable ---> [ PyListObject Header | ob_size | allocated ]
                                           │
                                           ▼ (Array of pointers)
                                  [ ptr0 | ptr1 | ptr2 | ptr3 ]
                                     │      │      │      │
                                     ▼      ▼      ▼      ▼
                                   PyObj  PyObj  PyObj  PyObj
```

### 2. Over-Allocation & Amortized $O(1)$ Append

When a list grows, CPython does not allocate space for just one element (which would make `.append()` $O(N)$ due to constant realloc calls). It allocates extra headroom using an over-allocation growth formula:

$$\text{new\_allocated} \approx \text{new\_size} + (\text{new\_size} \gg 3) + (\text{new\_size} < 9 \,?\, 3 : 6)$$

**Time Complexity:**
- `.append()`: $O(1)$ amortized
- `.insert(0, val)` or `.pop(0)`: $O(N)$ because every pointer in the internal array must be shifted in memory

### 3. Slice Notation Internals: [start:stop:step]

A slice `s[start:stop:step]` creates a slice object: `slice(start, stop, step)`.

**Step arithmetic:**
- **Positive step**: starts at start, increments by step, stops before reaching stop
- **Negative step**: starts at start, decrements by abs(step), stops before reaching stop (e.g., `[::-1]` flips the pointer array)

**Important:** Slicing a list always creates a new list containing a shallow copy of the original pointer addresses.

### 2. Over-Allocation & Amortized $O(1)$ Append

When a list grows, CPython does not allocate space for just one element (which would make `.append()` $O(N)$ due to constant realloc calls). It allocates extra headroom using an over-allocation growth formula:

$$\text{new\_allocated} \approx \text{new\_size} + (\text{new\_size} \gg 3) + (\text{new\_size} < 9 \,?\, 3 : 6)$$

**Time Complexity:**
- `.append()`: $O(1)$ amortized
- `.insert(0, val)` or `.pop(0)`: $O(N)$ because every pointer in the internal array must be shifted in memory

### 3. Slice Notation Internals: [start:stop:step]

A slice `s[start:stop:step]` creates a slice object: `slice(start, stop, step)`.

**Step arithmetic:**
- **Positive step**: starts at start, increments by step, stops before reaching stop
- **Negative step**: starts at start, decrements by abs(step), stops before reaching stop (e.g., `[::-1]` flips the pointer array)

**Important:** Slicing a list always creates a new list containing a shallow copy of the original pointer addresses.

## Constraints

Implement these exercises without using built-in helper methods like `.reverse()`, `.rotate()`, or importing `collections.deque`.

## Exercise 1: In-Place Array Reversal (Two-Pointer Technique)

Write a function `reverse_inplace(arr: list)` that reverses a list in-place ($O(1)$ auxiliary space) using manual index pointers `left` and `right`.

**Requirements:**
- Do not allocate a new list or use `arr[::-1]`
- Mutate the original list directly using pointer swapping: `arr[left], arr[right] = arr[right], arr[left]`

In [1]:
def reverse_inplace(arr:list, start=0, end=None):
    left = start
    right = len(arr)-1 if end==None else end
    print("before", arr)
    while left<right:
        arr[left], arr[right] = arr[right], arr[left]
        left+=1
        right-=1
    print("after",arr)
    return arr

In [2]:
reverse_inplace([3,2,1])

before [3, 2, 1]
after [1, 2, 3]


[1, 2, 3]

In [3]:
reverse_inplace([1,2,3,4,5,6],2,5)

before [1, 2, 3, 4, 5, 6]
after [1, 2, 6, 5, 4, 3]


[1, 2, 6, 5, 4, 3]

## Exercise 2: In-Place Array Rotation (The 3-Step Reversal Algorithm)

Rotate a list to the right by `k` steps in-place with $O(1)$ extra space.

### Why the 3-Reversal Approach?

**Time Complexity Comparison:**
- Repeatedly shifting elements: $O(n \times k)$ time
- The 3-reversal method: $O(n)$ time and $O(1)$ extra space

**Example with k = 3:**

| Approach | Work |
|----------|------|
| Shifting | Move elements 3 times → more work |
| Reversal | Each element is swapped only a constant number of times → $O(n)$ |

So the reversal algorithm is mainly about better time complexity while staying in-place.

In [4]:
def shift_array(arr:list,k:int):
    
    k = k % len(arr)
    
    arr = reverse_inplace(arr)
    
    arr = reverse_inplace(arr,0,k-1)
    
    arr = reverse_inplace(arr,k)
    
    print(arr)

In [5]:
shift_array([1,2,3,4,5,6,7],3)

before [1, 2, 3, 4, 5, 6, 7]
after [7, 6, 5, 4, 3, 2, 1]
before [7, 6, 5, 4, 3, 2, 1]
after [5, 6, 7, 4, 3, 2, 1]
before [5, 6, 7, 4, 3, 2, 1]
after [5, 6, 7, 1, 2, 3, 4]
[5, 6, 7, 1, 2, 3, 4]


## Exercise 3: Manual N-Dimensional Chunking via Slicing

Write a function `chunk_tensor_1d(data: list, chunk_size: int, step: int)` that splits a 1D list into overlapping or non-overlapping windows using manual slice arithmetic.

### Example:

```python
data = list(range(10))
chunk_tensor_1d(data, chunk_size=4, step=2)
```

**Expected output:**
```
[[0, 1, 2, 3], [2, 3, 4, 5], [4, 5, 6, 7], [6, 7, 8, 9]]
```

### Requirements:

Ensure the function cleanly handles truncating any trailing window that does not reach a full `chunk_size`.

In [ ]:
def chunk_overlap_arr(arr : list, chunk:int, step : int):
    new_arr=[]
    i=0
    while i<=len(arr)-chunk:
        new_arr.append(arr[i:i+chunk])
        i+=step
    print(new_arr)
        

In [ ]:
chunk_overlap_arr([1,2,3,4,5,6,7,8,10],3,1)

[[1, 2, 3], [2, 3, 4], [3, 4, 5], [4, 5, 6], [5, 6, 7], [6, 7, 8], [7, 8, 10]]
